# Dogecoin Price Prediction

## Project Information
- **Data Source**: DOGE-USD historical price data (Yahoo Finance / Kaggle)
- **Objective**: Predict Dogecoin closing prices using historical price features
- **Models Used**: Random Forest Regressor, Linear Regression
- **Best R² Score**: 99.61% (Linear Regression with scaled features)

## Dataset
The dataset contains daily cryptocurrency price data including:
- Open, High, Low, Close prices
- Adjusted Close price
- Trading Volume
- Date range: Historical DOGE-USD data


In [ ]:
%pip install -r ../requirements.txt


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
SEED = 42
np.random.seed(SEED)


In [ ]:
# Load and explore the dataset
df = pd.read_csv('DOGE-USD.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nDate range: {df['Date'].min()} to {df['Date'].max()}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.head()


In [ ]:
# Data preprocessing
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(by='Date').reset_index(drop=True)
df.dropna(inplace=True)

print(f"Dataset shape after preprocessing: {df.shape}")
print(f"\nData types:\n{df.dtypes}")
df.describe()


## Feature Engineering


In [ ]:
# Create derived features
df['Range'] = df['High'] - df['Low']  # Price range (volatility indicator)

# Prepare features and target
feature_cols = ['Open', 'High', 'Low', 'Range', 'Volume']
X = df[feature_cols]
y = df['Close']

print(f"Features: {feature_cols}")
print(f"Target: Close price")
print(f"\nFeature statistics:")
print(X.describe())


## Train-Test Split (Time Series Aware)


In [ ]:
# For time series, use chronological split (not random)
train_size = int(len(df) * 0.8)
X_train = X[:train_size]
X_test = X[train_size:]
y_train = y[:train_size]
y_test = y[train_size:]

print(f"Train set: {X_train.shape} ({df['Date'].iloc[0]} to {df['Date'].iloc[train_size-1]})")
print(f"Test set: {X_test.shape} ({df['Date'].iloc[train_size]} to {df['Date'].iloc[-1]})")


## Model 1: Random Forest Regressor (Baseline)


In [ ]:
# Train Random Forest model
rf_model = RandomForestRegressor(n_estimators=100, random_state=SEED)
rf_model.fit(X_train, y_train)

# Make predictions
rf_preds = rf_model.predict(X_test)

# Evaluate
rf_mse = mean_squared_error(y_test, rf_preds)
rf_r2 = r2_score(y_test, rf_preds)
rf_mae = mean_absolute_error(y_test, rf_preds)

print("Random Forest Results:")
print(f"R² Score: {rf_r2:.4f}")
print(f"MSE: {rf_mse:.6f}")
print(f"MAE: {rf_mae:.6f}")


## Model 2: Random Forest with Feature Scaling


In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train Random Forest with scaled features
rf_scaled = RandomForestRegressor(n_estimators=100, random_state=SEED)
rf_scaled.fit(X_train_scaled, y_train)

# Make predictions
rf_scaled_preds = rf_scaled.predict(X_test_scaled)

# Evaluate
rf_scaled_r2 = r2_score(y_test, rf_scaled_preds)
rf_scaled_mse = mean_squared_error(y_test, rf_scaled_preds)

print("Random Forest (Scaled) Results:")
print(f"R² Score: {rf_scaled_r2:.4f}")
print(f"MSE: {rf_scaled_mse:.6f}")


## Model 3: Linear Regression (Best Performance)


In [ ]:
# Train Linear Regression with scaled features
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Make predictions
lr_preds = lr_model.predict(X_test_scaled)

# Evaluate
lr_r2 = r2_score(y_test, lr_preds)
lr_mse = mean_squared_error(y_test, lr_preds)
lr_mae = mean_absolute_error(y_test, lr_preds)

print("Linear Regression Results:")
print(f"R² Score: {lr_r2:.4f}")
print(f"MSE: {lr_mse:.6f}")
print(f"MAE: {lr_mae:.6f}")


## Visualization: Predictions vs Actual


In [ ]:
# Plot predictions vs actual prices
test_dates = df['Date'].iloc[train_size:].values

plt.figure(figsize=(14, 6))
plt.plot(test_dates, y_test.values, label='Actual Prices', color='blue', linewidth=2)
plt.plot(test_dates, rf_preds, label=f'Random Forest (R²={rf_r2:.3f})', color='red', alpha=0.7)
plt.plot(test_dates, lr_preds, label=f'Linear Regression (R²={lr_r2:.3f})', color='green', alpha=0.7)
plt.xlabel('Date')
plt.ylabel('DOGE Price (USD)')
plt.title('Dogecoin Price Prediction: Actual vs Predicted')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
